<a href="https://colab.research.google.com/github/KashfiRashid/IAT460_A3/blob/main/A3_IAT460.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dissolve — IAT 460 A3
### Md Kashfi Or Rashid Pranta | 301540943
**Tool:** ComfyUI with Stable Diffusion 1.5 (DreamShaper)
**Concept:** A human silhouette gradually dissolves into organic-digital networks, exploring how technology quietly integrates into identity.

In [2]:
# Check what files are uploaded
!pip install moviepy Pillow -q

import os
files = sorted([f for f in os.listdir('/content/') if f.endswith(('.png', '.jpg', '.jpeg'))])
print(f"Found {len(files)} images:", files)
audio_files = [f for f in os.listdir('/content/') if f.endswith(('.wav', '.mp3'))]
print("Found audio:", audio_files)

Found 0 images: []
Found audio: []


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
# Check what's in your Generated_Images folder
drive_path = '/content/drive/MyDrive/Generated_Images'
files = sorted(os.listdir(drive_path))
print(f"Found {len(files)} files:", files)

Found 16 files: ['917.png', '918.png', '920.png', '922.png', '925.png', '927.png', '930.png', '932.png', '935.png', '937.png', '940.png', '942.png', '945.png', '947.png', '950.png', '955.png']


## Step 2: Upscale images to 1024x1024 and build video with crossfade transitions

In [5]:
from moviepy.editor import *
from PIL import Image
import numpy as np
import os

drive_path = '/content/drive/MyDrive/Generated_Images'

# Image order (by seed, matching cinematic arc)
image_order = ['917.png', '918.png', '920.png', '922.png', '925.png', '927.png',
               '930.png', '932.png', '935.png', '937.png', '940.png', '942.png',
               '945.png', '947.png', '950.png', '955.png']

# Upscale all images to 1024x1024
print("Upscaling images to 1024x1024...")
os.makedirs('/content/upscaled', exist_ok=True)
for fname in image_order:
    img = Image.open(f"{drive_path}/{fname}").resize((1024, 1024), Image.LANCZOS)
    img.save(f"/content/upscaled/{fname}")
print("Done!")

# Create video with crossfade transitions
print("Building video with crossfades...")
duration_per_image = 55 / len(image_order)  # ~3.4 sec each, 55 sec total

clips = []
for fname in image_order:
    clip = ImageClip(f"/content/upscaled/{fname}").set_duration(duration_per_image)
    clips.append(clip)

# Crossfade between each image
crossfade = 1.5  # seconds of overlap
final_clips = [clips[0]]
for i in range(1, len(clips)):
    final_clips.append(clips[i].set_start(i * duration_per_image - i * crossfade).crossfadein(crossfade))

content = CompositeVideoClip(final_clips)
content = content.subclip(0, 55)

# Add 3-second fade to black at end
content = content.fadeout(3)

# Create credits slide
from PIL import ImageDraw, ImageFont
cred_img = Image.new('RGB', (1024, 1024), (10, 10, 10))
draw = ImageDraw.Draw(cred_img)
try:
    font_l = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 40)
    font_m = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", 30)
    font_s = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", 24)
except:
    font_l = ImageFont.load_default()
    font_m = font_l
    font_s = font_l

def centered(text, y, font, fill=(255,255,255)):
    bbox = draw.textbbox((0,0), text, font=font)
    x = (1024 - (bbox[2]-bbox[0])) // 2
    draw.text((x, y), text, font=font, fill=fill)

centered("Md Kashfi Or Rashid Pranta", 350, font_l)
centered("IAT 460 Generative AI, Spring 2026", 420, font_m)
centered("School of Interactive Arts & Technology (SIAT)", 490, font_s, (180,180,180))
centered("Simon Fraser University", 525, font_s, (180,180,180))

cred_img.save("/content/credits.png")
credits_clip = ImageClip("/content/credits.png").set_duration(5)

# Combine credits + content
final = concatenate_videoclips([credits_clip, content])

# Add audio if available
audio_files = [f for f in os.listdir(drive_path) if f.endswith(('.wav', '.mp3'))]
if audio_files:
    audio = AudioFileClip(f"{drive_path}/{audio_files[0]}").subclip(0, min(60, final.duration))
    audio = audio.audio_fadeout(3)
    final = final.set_audio(audio)
    print(f"Audio added: {audio_files[0]}")
else:
    print("NO AUDIO FOUND - upload audio.wav to your Generated_Images folder!")

# Export
print("Exporting final video...")
final.write_videofile("/content/Dissolve_Final.mp4",
                      fps=24, codec="libx264",
                      audio_codec="aac", bitrate="5000k")

print(f"\nVideo duration: {final.duration:.1f} seconds")
print("DONE! Video saved to /content/Dissolve_Final.mp4")

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



Upscaling images to 1024x1024...
Done!
Building video with crossfades...
NO AUDIO FOUND - upload audio.wav to your Generated_Images folder!
Exporting final video...
Moviepy - Building video /content/Dissolve_Final.mp4.
Moviepy - Writing video /content/Dissolve_Final.mp4



Moviepy - Done !
Moviepy - video ready /content/Dissolve_Final.mp4

Video duration: 60.0 seconds
DONE! Video saved to /content/Dissolve_Final.mp4


In [6]:
from google.colab import files
files.download("/content/Dissolve_Final.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 3: Generate 300+ frames via img2img chaining for smooth animation

In [7]:
# Install and load model
!pip install diffusers transformers accelerate -q

import torch
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline
from PIL import Image
import os

model_id = "Lykon/dreamshaper-8"
pipe_txt2img = StableDiffusionPipeline.from_pretrained(
    model_id, torch_dtype=torch.float16, safety_checker=None
).to("cuda")

pipe_img2img = StableDiffusionImg2ImgPipeline(
    vae=pipe_txt2img.vae, text_encoder=pipe_txt2img.text_encoder,
    tokenizer=pipe_txt2img.tokenizer, unet=pipe_txt2img.unet,
    scheduler=pipe_txt2img.scheduler, safety_checker=None, feature_extractor=None
).to("cuda")

os.makedirs("/content/frames", exist_ok=True)

# Generate base silhouette
generator = torch.Generator(device="cuda").manual_seed(917)
base = pipe_txt2img(
    prompt="dark noir portrait, person facing forward, head and upper body visible as dark silhouette, very dim subtle rim light on edges only, almost entirely black image, dark moody void, matte, cinematic, abstract",
    negative_prompt="bright background, spotlight, backlight, white light, blue, colorful, detailed face, side view, profile view, text, watermark, multiple people, low quality, deformed, halo, ring",
    width=512, height=512, num_inference_steps=30, guidance_scale=8.0, generator=generator
).images[0]
base.save("/content/frames/frame_0000.png")
print("Base silhouette generated!")

# Generate 300 frames across 4 cinematic phases
phases = [
    {"prompt": "dark moody portrait, person facing forward, dark silhouette, faint heartbeat glow in chest, black void, minimal, cinematic, noir, matte",
     "strength": 0.35, "guidance": 7.0, "frames": 75},
    {"prompt": "dark figure with organic roots and veins growing outward from body, mycelium tendrils branching from silhouette, faint bioluminescent teal and amber traces, neural pathways forming, dark void background, cinematic, moody",
     "strength": 0.45, "guidance": 7.5, "frames": 80},
    {"prompt": "human figure dissolving into dense network of roots circuits and data streams, bioluminescent neural pathways everywhere, digital signals merging with organic tissue, glowing teal and amber veins, overwhelming complexity, dark void, cinematic",
     "strength": 0.55, "guidance": 8.0, "frames": 80},
    {"prompt": "abstract dissolving particles fading into darkness, remnants of organic digital network, dark void consuming all form, soft ambient glow fading away, minimal, nearly black, cinematic",
     "strength": 0.50, "guidance": 7.0, "frames": 65}
]

negative = "bright, white background, text, watermark, low quality, deformed, blurry, cheerful, cartoon"
current_img = base.copy()
frame_count = 1

for phase_idx, phase in enumerate(phases):
    print(f"\n=== Phase {phase_idx+1}/4: {phase['frames']} frames ===")
    for i in range(phase["frames"]):
        generator = torch.Generator(device="cuda").manual_seed(frame_count + 42)
        result = pipe_img2img(
            prompt=phase["prompt"], negative_prompt=negative,
            image=current_img, strength=phase["strength"],
            guidance_scale=phase["guidance"], num_inference_steps=20,
            generator=generator
        ).images[0]
        result.save(f"/content/frames/frame_{frame_count:04d}.png")
        current_img = result
        frame_count += 1
        if frame_count % 50 == 0:
            print(f"  Frame {frame_count}/300")

print(f"\n=== All {frame_count} frames generated! ===")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



model_index.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

CLIPFeatureExtractor appears to have been deprecated in transformers. Using CLIPImageProcessor instead.


...

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  Frame 300/300


  0%|          | 0/10 [00:00<?, ?it/s]


=== All 301 frames generated! ===


## Step 4: Upscale to 1024x1024, add credits + audio, export final mp4

In [10]:
from moviepy.editor import *
from PIL import Image, ImageDraw, ImageFont
import os

print("Upscaling 301 frames to 1024x1024...")
os.makedirs("/content/frames_1024", exist_ok=True)
for i in range(301):
    img = Image.open(f"/content/frames/frame_{i:04d}.png")
    img = img.resize((1024, 1024), Image.LANCZOS)
    img.save(f"/content/frames_1024/frame_{i:04d}.png")
    if i % 50 == 0:
        print(f"  {i}/301")
print("  301/301 done!")

print("\nBuilding video...")
frame_files = [f"/content/frames_1024/frame_{i:04d}.png" for i in range(301)]
content = ImageSequenceClip(frame_files, fps=6)
content = content.subclip(0, min(55, content.duration))
content = content.fadeout(3)

print("Creating credits slide...")
cred_img = Image.new('RGB', (1024, 1024), (10, 10, 10))
draw = ImageDraw.Draw(cred_img)
font_l = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 40)
font_m = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", 30)
font_s = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", 24)

def centered(text, y, font, fill=(255,255,255)):
    bbox = draw.textbbox((0,0), text, font=font)
    x = (1024 - (bbox[2]-bbox[0])) // 2
    draw.text((x, y), text, font=font, fill=fill)

centered("Md Kashfi Or Rashid Pranta", 350, font_l)
centered("IAT 460 Generative AI, Spring 2026", 420, font_m)
centered("School of Interactive Arts & Technology (SIAT)", 490, font_s, (180,180,180))
centered("Simon Fraser University", 525, font_s, (180,180,180))
cred_img.save("/content/credits.png")
credits_clip = ImageClip("/content/credits.png").set_duration(5)

final = concatenate_videoclips([credits_clip, content])

drive_path = '/content/drive/MyDrive/Generated_Images'
audio_files = [f for f in os.listdir(drive_path) if f.endswith(('.wav', '.mp3'))]
if audio_files:
    audio = AudioFileClip(f"{drive_path}/{audio_files[0]}")
    audio = audio.subclip(0, min(audio.duration, final.duration))
    audio = audio.audio_fadeout(3)
    final = final.set_audio(audio)
    print(f"Audio added: {audio_files[0]}")
else:
    print("NO AUDIO - upload audio file to Generated_Images folder on Drive!")

print("\nExporting...")
final.write_videofile("/content/Dissolve_Final.mp4", fps=24, codec="libx264", audio_codec="aac", bitrate="5000k")
print(f"\nDuration: {final.duration:.1f} seconds")
print("DONE!")

Upscaling 301 frames to 1024x1024...
  0/301
  50/301
  100/301
  150/301
  200/301
  250/301
  300/301
  301/301 done!

Building video...
Creating credits slide...
NO AUDIO - upload audio file to Generated_Images folder on Drive!

Exporting...
Moviepy - Building video /content/Dissolve_Final.mp4.
Moviepy - Writing video /content/Dissolve_Final.mp4



Moviepy - Done !
Moviepy - video ready /content/Dissolve_Final.mp4

Duration: 55.2 seconds
DONE!


In [13]:
from moviepy.editor import *

video = VideoFileClip("/content/Dissolve_Final.mp4")
drive_path = '/content/drive/MyDrive/Generated_Images'

import os
audio_files = [f for f in os.listdir(drive_path) if f.endswith(('.wav', '.mp3'))]
print("Audio files found:", audio_files)

if audio_files:
    audio = AudioFileClip(f"{drive_path}/{audio_files[0]}")
    audio = audio.subclip(0, min(audio.duration, video.duration))
    audio = audio.audio_fadeout(3)
    final = video.set_audio(audio)
    final.write_videofile("/content/Dissolve_Final_Audio.mp4", fps=24, codec="libx264", audio_codec="aac", bitrate="5000k")
    print("DONE WITH AUDIO!")
else:
    print("Still no audio file found!")

Audio files found: ['audio.wav']
Moviepy - Building video /content/Dissolve_Final_Audio.mp4.
MoviePy - Writing audio in Dissolve_Final_AudioTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/Dissolve_Final_Audio.mp4



t: 100%|█████████▉| 1324/1325 [01:31<00:00, 14.83it/s, now=None]WARNING:py.warnings:/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:123: UserWarning: Warning: in file /content/Dissolve_Final.mp4, 3145728 bytes wanted but 0 bytes read,at frame 1324/1325, at time 55.17/55.17 sec. Using the last valid frame instead.
  warnings.warn("Warning: in file %s, "%(self.filename)+



Moviepy - Done !
Moviepy - video ready /content/Dissolve_Final_Audio.mp4
DONE WITH AUDIO!


In [14]:
from google.colab import files
files.download("/content/Dissolve_Final_Audio.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>